# Corpus, Document, Vocabulary and Word in NLP

This notebook demonstrates the four core NLP concepts with pure Python (no installs needed).
See `theory.md` for the explanations.

## 1. Corpus and Documents
A **corpus** is a collection of **documents**. Here each document is one short review.

In [1]:
corpus = [
    "I love NLP.",
    "NLP loves data!",
    "I love data, and I love Python.",
    "Machine learning needs data.",
]

print("Corpus size (number of documents):", len(corpus))
for i, doc in enumerate(corpus):
    print(f"Document {i}: {doc!r}")

Corpus size (number of documents): 4
Document 0: 'I love NLP.'
Document 1: 'NLP loves data!'
Document 2: 'I love data, and I love Python.'
Document 3: 'Machine learning needs data.'


## 2. Words (Tokens)
**Tokenization** splits each document into tokens. We lowercase and strip punctuation with a simple regex.

In [2]:
import re

def tokenize(text):
    return re.findall(r"[a-z0-9']+", text.lower())

tokenized_corpus = [tokenize(doc) for doc in corpus]

for i, tokens in enumerate(tokenized_corpus):
    print(f"Document {i} -> {tokens}")

Document 0 -> ['i', 'love', 'nlp']
Document 1 -> ['nlp', 'loves', 'data']
Document 2 -> ['i', 'love', 'data', 'and', 'i', 'love', 'python']
Document 3 -> ['machine', 'learning', 'needs', 'data']


### Token occurrences vs. word types

In [3]:
all_tokens = [tok for doc in tokenized_corpus for tok in doc]

print("Total tokens (occurrences):", len(all_tokens))
print("Unique word types         :", len(set(all_tokens)))

Total tokens (occurrences): 17
Unique word types         : 10


## 3. Vocabulary
The **vocabulary** is the set of unique tokens in the corpus. We map each word to an integer ID.

In [4]:
vocab = sorted(set(all_tokens))
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

print("Vocabulary size |V| =", len(vocab))
print("Vocabulary:", vocab)
print()
print("word2idx:", word2idx)

Vocabulary size |V| = 10
Vocabulary: ['and', 'data', 'i', 'learning', 'love', 'loves', 'machine', 'needs', 'nlp', 'python']

word2idx: {'and': 0, 'data': 1, 'i': 2, 'learning': 3, 'love': 4, 'loves': 5, 'machine': 6, 'needs': 7, 'nlp': 8, 'python': 9}


### Encoding documents as ID sequences

In [5]:
encoded = [[word2idx[tok] for tok in doc] for doc in tokenized_corpus]

for doc, ids in zip(tokenized_corpus, encoded):
    print(doc, "->", ids)

['i', 'love', 'nlp'] -> [2, 4, 8]
['nlp', 'loves', 'data'] -> [8, 5, 1]
['i', 'love', 'data', 'and', 'i', 'love', 'python'] -> [2, 4, 1, 0, 2, 4, 9]
['machine', 'learning', 'needs', 'data'] -> [6, 3, 7, 1]


### Word frequencies in the corpus

In [6]:
from collections import Counter

word_freq = Counter(all_tokens)
for word, count in word_freq.most_common():
    print(f"{word:10s} {count}")

i          3
love       3
data       3
nlp        2
loves      1
and        1
python     1
machine    1
learning   1
needs      1


## 4. Special tokens and Out-of-Vocabulary (OOV) words
Words never seen in the training corpus are mapped to `<UNK>`. `<PAD>` is used to make sequences equal length.

In [7]:
special_tokens = ["<PAD>", "<UNK>"]
vocab_with_special = special_tokens + vocab

word2idx = {w: i for i, w in enumerate(vocab_with_special)}
idx2word = {i: w for w, i in word2idx.items()}

def encode(text):
    return [word2idx.get(tok, word2idx["<UNK>"]) for tok in tokenize(text)]

def decode(ids):
    return " ".join(idx2word[i] for i in ids)

new_sentence = "I love deep learning with data"   # 'deep' and 'with' are OOV
ids = encode(new_sentence)

print("Sentence :", new_sentence)
print("Encoded  :", ids)
print("Decoded  :", decode(ids))

Sentence : I love deep learning with data
Encoded  : [4, 6, 1, 5, 1, 3]
Decoded  : i love <UNK> learning <UNK> data


In [8]:
def pad(sequences, max_len=None):
    max_len = max_len or max(len(s) for s in sequences)
    return [s + [word2idx["<PAD>"]] * (max_len - len(s)) for s in sequences]

batch = [encode(doc) for doc in corpus]
for row in pad(batch):
    print(row)

[4, 6, 10, 0, 0, 0, 0]
[10, 7, 3, 0, 0, 0, 0]
[4, 6, 3, 2, 4, 6, 11]
[8, 5, 9, 3, 0, 0, 0]


## 5. Relationship: Corpus > Document > Token, and Vocabulary
`|V| <= total tokens`. The Type-Token Ratio measures lexical diversity.

In [9]:
total_tokens = len(all_tokens)
vocab_size = len(set(all_tokens))

print("Documents    :", len(corpus))
print("Total tokens :", total_tokens)
print("Vocabulary   :", vocab_size)
print("Type-Token Ratio:", round(vocab_size / total_tokens, 3))

Documents    : 4
Total tokens : 17
Vocabulary   : 10
Type-Token Ratio: 0.588


## 6. Using the vocabulary: Bag of Words vector
Each document becomes a vector with one dimension per vocabulary word, so the vocabulary decides the vector length.

In [10]:
real_vocab = vocab  # without special tokens

def bag_of_words(tokens):
    counts = Counter(tokens)
    return [counts.get(w, 0) for w in real_vocab]

print("Columns:", real_vocab)
for i, doc in enumerate(tokenized_corpus):
    print(f"Doc {i}:", bag_of_words(doc))

Columns: ['and', 'data', 'i', 'learning', 'love', 'loves', 'machine', 'needs', 'nlp', 'python']
Doc 0: [0, 0, 1, 0, 1, 0, 0, 0, 1, 0]
Doc 1: [0, 1, 0, 0, 0, 1, 0, 0, 1, 0]
Doc 2: [1, 1, 2, 0, 2, 0, 0, 0, 0, 1]
Doc 3: [0, 1, 0, 1, 0, 0, 1, 1, 0, 0]


## 7. Document Frequency and TF-IDF (simple version)
This needs all three: documents (TF), corpus (DF, N) and vocabulary.

In [11]:
import math

N = len(tokenized_corpus)
df = {w: sum(1 for doc in tokenized_corpus if w in doc) for w in real_vocab}

def tfidf(tokens):
    counts = Counter(tokens)
    return {w: round(counts[w] * math.log(N / df[w]), 3) for w in counts}

print("Document frequency:", df)
print()
for i, doc in enumerate(tokenized_corpus):
    print(f"Doc {i}:", tfidf(doc))

Document frequency: {'and': 1, 'data': 3, 'i': 2, 'learning': 1, 'love': 2, 'loves': 1, 'machine': 1, 'needs': 1, 'nlp': 2, 'python': 1}

Doc 0: {'i': 0.693, 'love': 0.693, 'nlp': 0.693}
Doc 1: {'nlp': 0.693, 'loves': 1.386, 'data': 0.288}
Doc 2: {'i': 1.386, 'love': 1.386, 'data': 0.288, 'and': 1.386, 'python': 1.386}
Doc 3: {'machine': 1.386, 'learning': 1.386, 'needs': 1.386, 'data': 0.288}


## 8. Building a vocabulary with a minimum frequency
In real projects we drop rare words to keep the vocabulary small (they become `<UNK>`).

In [12]:
min_freq = 2
filtered_vocab = ["<PAD>", "<UNK>"] + sorted(w for w, c in word_freq.items() if c >= min_freq)

print("Original |V|:", len(real_vocab))
print("Filtered |V| (min_freq=2):", len(filtered_vocab) - 2)
print("Kept words:", filtered_vocab[2:])

Original |V|: 10
Filtered |V| (min_freq=2): 4
Kept words: ['data', 'i', 'love', 'nlp']


## 9. Optional: the same idea with scikit-learn
`CountVectorizer` builds the vocabulary from a corpus automatically (runs only if scikit-learn is installed).

In [13]:
try:
    from sklearn.feature_extraction.text import CountVectorizer

    cv = CountVectorizer()
    X = cv.fit_transform(corpus)          # fit = build vocabulary from the corpus

    print("Vocabulary:", cv.vocabulary_)
    print("Feature names:", list(cv.get_feature_names_out()))
    print("Matrix shape (documents x |V|):", X.shape)
    print(X.toarray())
except ImportError:
    print("scikit-learn not installed. Run: pip install scikit-learn")

Vocabulary: {'love': 3, 'nlp': 7, 'loves': 4, 'data': 1, 'and': 0, 'python': 8, 'machine': 5, 'learning': 2, 'needs': 6}
Feature names: ['and', 'data', 'learning', 'love', 'loves', 'machine', 'needs', 'nlp', 'python']
Matrix shape (documents x |V|): (4, 9)
[[0 0 0 1 0 0 0 1 0]
 [0 1 0 0 1 0 0 1 0]
 [1 1 0 2 0 0 0 0 1]
 [0 1 1 0 0 1 1 0 0]]


## Summary

| Concept | In this notebook |
|---------|------------------|
| Word / Token | items in `tokenized_corpus` |
| Document | each string in `corpus` |
| Corpus | the whole `corpus` list |
| Vocabulary | `vocab` / `word2idx` (unique words with IDs) |